# STEP 9 - Notebook Relevance and Archival Workflow

This notebook identifies irrelevant notebooks using transparent scoring rules and archives them safely with full audit and restore support.

- Supports dry-run previews
- Preserves folder structure in archive
- Writes audit logs and restore manifest
- Includes unit test scaffolding for reproducibility

## 1. Set Up Workspace and Paths
Import libraries and define core paths and safety flags.

In [2]:
from pathlib import Path
import json
import shutil
import re
from datetime import datetime, timezone
from typing import Dict, List, Tuple

import pandas as pd

# Resolve project root robustly even when notebook runs from notebooks/.
if (Path.cwd() / "config" / "settings.py").exists():
    WORKSPACE = Path.cwd()
elif (Path.cwd().parent / "config" / "settings.py").exists():
    WORKSPACE = Path.cwd().parent
else:
    WORKSPACE = Path.cwd()

NOTEBOOKS_DIR = WORKSPACE / "notebooks"
ARCHIVE_ROOT = WORKSPACE / "archive" / "notebooks"
AUDIT_DIR = WORKSPACE / "outputs" / "notebook_archive_audit"

DRY_RUN = True
ALLOW_OVERWRITE = False

RUN_TS = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ARCHIVE_DIR = ARCHIVE_ROOT / f"archived_{RUN_TS}"
AUDIT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Workspace: {WORKSPACE}")
print(f"Notebooks dir: {NOTEBOOKS_DIR}")
print(f"Archive root: {ARCHIVE_ROOT}")
print(f"Run archive dir: {RUN_ARCHIVE_DIR}")
print(f"DRY_RUN={DRY_RUN}, ALLOW_OVERWRITE={ALLOW_OVERWRITE}")

Workspace: /Users/vinayksharma/AirDnd/cti_recommender
Notebooks dir: /Users/vinayksharma/AirDnd/cti_recommender/notebooks
Archive root: /Users/vinayksharma/AirDnd/cti_recommender/archive/notebooks
Run archive dir: /Users/vinayksharma/AirDnd/cti_recommender/archive/notebooks/archived_20260308_075043
DRY_RUN=True, ALLOW_OVERWRITE=False


## 2. Scan Notebook Files Recursively
Discover notebook files while excluding archive locations.

In [3]:
def discover_notebooks(root: Path) -> pd.DataFrame:
    rows = []
    for p in root.rglob("*.ipynb"):
        rp = p.relative_to(WORKSPACE)
        if "archive/notebooks" in str(rp).replace("\\", "/"):
            continue
        st = p.stat()
        rows.append(
            {
                "path": str(rp),
                "abs_path": str(p),
                "size_bytes": int(st.st_size),
                "mtime": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
            }
        )
    return pd.DataFrame(rows).sort_values("path").reset_index(drop=True)

scan_df = discover_notebooks(WORKSPACE)
print(f"Discovered notebooks: {len(scan_df)}")
scan_df.head(10)

Discovered notebooks: 8


,path,abs_path,size_bytes,mtime
0,notebooks/STEP_0_Support_Notebook.ipynb,/Users/vinayksharma/AirDnd/cti_recommender/not...,27473,2026-03-08T05:30:47.608813+00:00
1,notebooks/STEP_1_Data_Ingestion_Pipeline.ipynb,/Users/vinayksharma/AirDnd/cti_recommender/not...,85874,2026-03-08T05:31:59.739017+00:00
2,notebooks/STEP_2_EDA_Analysis.ipynb,/Users/vinayksharma/AirDnd/cti_recommender/not...,75796,2026-03-08T05:32:33.841383+00:00
3,notebooks/STEP_3_Compute_Features.ipynb,/Users/vinayksharma/AirDnd/cti_recommender/not...,34469,2026-03-08T05:32:58.969251+00:00
4,notebooks/STEP_4_Feature_Engineering_Labels.ipynb,/Users/vinayksharma/AirDnd/cti_recommender/not...,59768,2026-03-08T05:33:22.714009+00:00
5,notebooks/STEP_5_Model_Training_And_Evaluation...,/Users/vinayksharma/AirDnd/cti_recommender/not...,114764,2026-03-08T05:42:31.427696+00:00
6,notebooks/STEP_8_Advanced_Models_GraphBased.ipynb,/Users/vinayksharma/AirDnd/cti_recommender/not...,47653,2026-02-26T16:21:29.190069+00:00
7,notebooks/STEP_9_Notebook_Archival_Workflow.ipynb,/Users/vinayksharma/AirDnd/cti_recommender/not...,17282,2026-03-08T07:50:38.388695+00:00


## 3. Extract Notebook Text and Metadata
Read notebook JSON, aggregate cell text, and capture metadata fields for scoring.

In [4]:
def extract_notebook_payload(abs_path: Path) -> Dict:
    try:
        obj = json.loads(abs_path.read_text(encoding="utf-8"))
    except Exception:
        return {
            "markdown_text": "",
            "code_text": "",
            "kernel": "",
            "tags": "",
            "parse_error": 1,
        }

    md_chunks: List[str] = []
    code_chunks: List[str] = []
    tags: List[str] = []
    kernel = str(obj.get("metadata", {}).get("kernelspec", {}).get("name", ""))

    for cell in obj.get("cells", []):
        ctype = cell.get("cell_type", "")
        src = "".join(cell.get("source", [])) if isinstance(cell.get("source", []), list) else str(cell.get("source", ""))
        if ctype == "markdown":
            md_chunks.append(src)
        elif ctype == "code":
            code_chunks.append(src)
        ctags = cell.get("metadata", {}).get("tags", [])
        if isinstance(ctags, list):
            tags.extend([str(t) for t in ctags])

    return {
        "markdown_text": "\n".join(md_chunks),
        "code_text": "\n".join(code_chunks),
        "kernel": kernel,
        "tags": ",".join(sorted(set(tags))),
        "parse_error": 0,
    }

extracted_rows = []
for _, r in scan_df.iterrows():
    payload = extract_notebook_payload(Path(r["abs_path"]))
    extracted_rows.append({**r.to_dict(), **payload})

nb_df = pd.DataFrame(extracted_rows)
nb_df["full_text"] = (nb_df["markdown_text"].fillna("") + "\n" + nb_df["code_text"].fillna("")).str.lower()
print(f"Parsed notebooks: {len(nb_df)}")
nb_df[["path", "kernel", "parse_error"]].head(10)

Parsed notebooks: 8


,path,kernel,parse_error
0,notebooks/STEP_0_Support_Notebook.ipynb,python3,0
1,notebooks/STEP_1_Data_Ingestion_Pipeline.ipynb,python3,0
2,notebooks/STEP_2_EDA_Analysis.ipynb,python3,0
3,notebooks/STEP_3_Compute_Features.ipynb,python3,0
4,notebooks/STEP_4_Feature_Engineering_Labels.ipynb,python3,0
5,notebooks/STEP_5_Model_Training_And_Evaluation...,python3,0
6,notebooks/STEP_8_Advanced_Models_GraphBased.ipynb,python3,0
7,notebooks/STEP_9_Notebook_Archival_Workflow.ipynb,python3,0


## 4. Define Relevance Rules and Scoring
Use weighted keyword signals, recency effects, and path exclusions to compute notebook relevance score.

In [5]:
RULES = {
    "high_value": {
        "pattern": r"(research question|rq1|rq2|rq3|lambdamart|rgcn|diffusion|ablation|temporal split|70/30)",
        "weight": 3.0,
    },
    "pipeline_step": {
        "pattern": r"(step_[0-9]|data ingestion|feature engineering|model training|evaluation)",
        "weight": 2.0,
    },
    "legacy_or_tmp": {
        "pattern": r"(untitled|scratch|backup|old|deprecated|legacy)",
        "weight": -2.5,
    },
}

PATH_EXCLUSIONS = [r"^notebooks/cache/", r"^archive/"]
RECENCY_PENALTY_DAYS = 365 * 2
RECENCY_PENALTY = -0.8


def score_notebook(row: pd.Series) -> Tuple[float, List[str]]:
    path = str(row["path"]).replace("\\", "/")
    text = str(row.get("full_text", ""))
    score = 0.0
    reasons: List[str] = []

    for ex in PATH_EXCLUSIONS:
        if re.search(ex, path):
            reasons.append("path_excluded")
            return -999.0, reasons

    for rule_name, spec in RULES.items():
        if re.search(spec["pattern"], text):
            score += float(spec["weight"])
            reasons.append(rule_name)

    mtime = pd.to_datetime(row.get("mtime"), errors="coerce", utc=True)
    if pd.notna(mtime):
        age_days = (datetime.now(timezone.utc) - mtime.to_pydatetime()).days
        if age_days > RECENCY_PENALTY_DAYS:
            score += RECENCY_PENALTY
            reasons.append("stale_penalty")

    return score, reasons

scores = nb_df.apply(lambda r: score_notebook(r), axis=1)
nb_df["relevance_score"] = scores.apply(lambda x: x[0])
nb_df["reasons"] = scores.apply(lambda x: ";".join(x[1]))
nb_df[["path", "relevance_score", "reasons"]].sort_values("relevance_score").head(15)

,path,relevance_score,reasons
0,notebooks/STEP_0_Support_Notebook.ipynb,-2.5,legacy_or_tmp
2,notebooks/STEP_2_EDA_Analysis.ipynb,-0.5,pipeline_step;legacy_or_tmp
1,notebooks/STEP_1_Data_Ingestion_Pipeline.ipynb,2.0,pipeline_step
3,notebooks/STEP_3_Compute_Features.ipynb,2.0,pipeline_step
5,notebooks/STEP_5_Model_Training_And_Evaluation...,2.5,high_value;pipeline_step;legacy_or_tmp
6,notebooks/STEP_8_Advanced_Models_GraphBased.ipynb,2.5,high_value;pipeline_step;legacy_or_tmp
7,notebooks/STEP_9_Notebook_Archival_Workflow.ipynb,2.5,high_value;pipeline_step;legacy_or_tmp
4,notebooks/STEP_4_Feature_Engineering_Labels.ipynb,5.0,high_value;pipeline_step


## 5. Classify Notebooks as Relevant or Irrelevant
Apply thresholding and keep decision reasons for traceability.

In [6]:
THRESHOLD = 1.0
nb_df["decision"] = nb_df["relevance_score"].apply(lambda s: "irrelevant" if s < THRESHOLD else "relevant")

summary = nb_df["decision"].value_counts().rename_axis("decision").reset_index(name="count")
summary

,decision,count
0,relevant,6
1,irrelevant,2


## 6. Preview Archive Plan
Dry-run preview with destination path and conflict checks before file operations.

In [7]:
archive_plan = nb_df[nb_df["decision"] == "irrelevant"].copy()
archive_plan["archive_path"] = archive_plan["path"].apply(lambda p: str((RUN_ARCHIVE_DIR / p).as_posix()))
archive_plan["archive_exists"] = archive_plan["archive_path"].apply(lambda p: Path(p).exists())

print(f"Archive candidates: {len(archive_plan)}")
archive_plan[["path", "relevance_score", "reasons", "archive_path", "archive_exists"]].head(20)

Archive candidates: 2


,path,relevance_score,reasons,archive_path,archive_exists
0,notebooks/STEP_0_Support_Notebook.ipynb,-2.5,legacy_or_tmp,/Users/vinayksharma/AirDnd/cti_recommender/arc...,False
2,notebooks/STEP_2_EDA_Analysis.ipynb,-0.5,pipeline_step;legacy_or_tmp,/Users/vinayksharma/AirDnd/cti_recommender/arc...,False


## 7. Archive Irrelevant Notebooks Safely
Move files into timestamped archive while preserving structure and handling collisions.

In [8]:
def _resolve_collision(dst: Path) -> Path:
    if not dst.exists() or ALLOW_OVERWRITE:
        return dst
    stem = dst.stem
    suffix = dst.suffix
    parent = dst.parent
    i = 1
    while True:
        cand = parent / f"{stem}__dup{i}{suffix}"
        if not cand.exists():
            return cand
        i += 1

moves: List[Dict] = []
for _, row in archive_plan.iterrows():
    src = WORKSPACE / row["path"]
    dst = RUN_ARCHIVE_DIR / row["path"]
    dst = _resolve_collision(dst)

    action = "dry-run"
    if not DRY_RUN:
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.move(str(src), str(dst))
        action = "moved"

    moves.append(
        {
            "source_path": str(src),
            "archive_path": str(dst),
            "score": float(row["relevance_score"]),
            "decision": row["decision"],
            "reasons": row["reasons"],
            "run_timestamp": RUN_TS,
            "action": action,
        }
    )

moves_df = pd.DataFrame(moves)
print(f"Planned/processed moves: {len(moves_df)}")
moves_df.head(10)

Planned/processed moves: 2


,source_path,archive_path,score,decision,reasons,run_timestamp,action
0,/Users/vinayksharma/AirDnd/cti_recommender/not...,/Users/vinayksharma/AirDnd/cti_recommender/arc...,-2.5,irrelevant,legacy_or_tmp,20260308_075043,dry-run
1,/Users/vinayksharma/AirDnd/cti_recommender/not...,/Users/vinayksharma/AirDnd/cti_recommender/arc...,-0.5,irrelevant,pipeline_step;legacy_or_tmp,20260308_075043,dry-run


## 8. Create Audit Log and Restore Manifest
Save CSV/JSON logs and provide restore helper for archived files.

In [9]:
audit_csv = AUDIT_DIR / f"archive_audit_{RUN_TS}.csv"
audit_json = AUDIT_DIR / f"archive_audit_{RUN_TS}.json"
manifest_json = AUDIT_DIR / f"restore_manifest_{RUN_TS}.json"

if len(moves_df) > 0:
    moves_df.to_csv(audit_csv, index=False)
    audit_json.write_text(moves_df.to_json(orient="records", indent=2), encoding="utf-8")
    manifest = moves_df[["source_path", "archive_path", "run_timestamp"]].to_dict(orient="records")
    manifest_json.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

print(f"Audit CSV: {audit_csv}")
print(f"Audit JSON: {audit_json}")
print(f"Restore manifest: {manifest_json}")


def restore_from_manifest(manifest_path: Path, dry_run: bool = True) -> pd.DataFrame:
    entries = json.loads(manifest_path.read_text(encoding="utf-8"))
    restored = []
    for e in entries:
        src = Path(e["archive_path"])
        dst = Path(e["source_path"])
        action = "dry-run"
        if not dry_run:
            dst.parent.mkdir(parents=True, exist_ok=True)
            if src.exists():
                shutil.move(str(src), str(dst))
                action = "restored"
            else:
                action = "missing_archive_file"
        restored.append({"archive_path": str(src), "source_path": str(dst), "action": action})
    return pd.DataFrame(restored)

# Example dry-run restore check (disabled by default)
# restore_preview = restore_from_manifest(manifest_json, dry_run=True)
# restore_preview.head()

Audit CSV: /Users/vinayksharma/AirDnd/cti_recommender/outputs/notebook_archive_audit/archive_audit_20260308_075043.csv
Audit JSON: /Users/vinayksharma/AirDnd/cti_recommender/outputs/notebook_archive_audit/archive_audit_20260308_075043.json
Restore manifest: /Users/vinayksharma/AirDnd/cti_recommender/outputs/notebook_archive_audit/restore_manifest_20260308_075043.json


## 9. Add Unit Tests for Classification and Archiving
Pytest examples for scoring, dry-run behavior, and archive/restore integrity.

In [10]:
pytest_template = r'''
import json
from pathlib import Path

import pandas as pd


def test_threshold_classification():
    df = pd.DataFrame({"relevance_score": [2.0, 0.2, -1.0]})
    decision = df["relevance_score"].apply(lambda s: "irrelevant" if s < 1.0 else "relevant")
    assert decision.tolist() == ["relevant", "irrelevant", "irrelevant"]


def test_dry_run_no_move(tmp_path: Path):
    src = tmp_path / "n1.ipynb"
    src.write_text("{}", encoding="utf-8")
    dst = tmp_path / "archive" / "n1.ipynb"
    dry_run = True
    if not dry_run:
        dst.parent.mkdir(parents=True, exist_ok=True)
        src.rename(dst)
    assert src.exists()
    assert not dst.exists()


def test_restore_manifest_roundtrip(tmp_path: Path):
    src = tmp_path / "src.ipynb"
    arc = tmp_path / "archive" / "src.ipynb"
    src.write_text("{}", encoding="utf-8")
    arc.parent.mkdir(parents=True, exist_ok=True)
    src.rename(arc)

    manifest = [{"source_path": str(src), "archive_path": str(arc), "run_timestamp": "ts"}]
    mpath = tmp_path / "manifest.json"
    mpath.write_text(json.dumps(manifest), encoding="utf-8")

    entries = json.loads(mpath.read_text(encoding="utf-8"))
    for e in entries:
        Path(e["source_path"]).parent.mkdir(parents=True, exist_ok=True)
        Path(e["archive_path"]).rename(Path(e["source_path"]))

    assert src.exists()
    assert not arc.exists()
'''

print(pytest_template)
print("\nSave this into tests/test_notebook_archival.py to execute with pytest.")


import json
from pathlib import Path

import pandas as pd


def test_threshold_classification():
    df = pd.DataFrame({"relevance_score": [2.0, 0.2, -1.0]})
    decision = df["relevance_score"].apply(lambda s: "irrelevant" if s < 1.0 else "relevant")
    assert decision.tolist() == ["relevant", "irrelevant", "irrelevant"]


def test_dry_run_no_move(tmp_path: Path):
    src = tmp_path / "n1.ipynb"
    src.write_text("{}", encoding="utf-8")
    dst = tmp_path / "archive" / "n1.ipynb"
    dry_run = True
    if not dry_run:
        dst.parent.mkdir(parents=True, exist_ok=True)
        src.rename(dst)
    assert src.exists()
    assert not dst.exists()


def test_restore_manifest_roundtrip(tmp_path: Path):
    src = tmp_path / "src.ipynb"
    arc = tmp_path / "archive" / "src.ipynb"
    src.write_text("{}", encoding="utf-8")
    arc.parent.mkdir(parents=True, exist_ok=True)
    src.rename(arc)

    manifest = [{"source_path": str(src), "archive_path": str(arc), "run_timestamp": "ts"}]
   